In [5]:
import pandas as pd

# Read the file, treating the first column as an unnamed index
df = pd.read_csv('cleaned_crmls_dataset.txt', index_col=0)

# Show the first few rows
print(df.head())

    ViewYN  PoolPrivateYN   Latitude   Longitude  LivingArea CountyOrParish  \
15    True          False  33.725080 -117.222302      2824.0      Riverside   
17    True          False  34.203479 -118.643567      2500.0    Los Angeles   
19   False          False  34.460368 -118.490755      2363.0    Los Angeles   
20   False          False  34.043218 -118.519477      3338.0    Los Angeles   
25   False          False  37.669495 -121.763793      2485.0        Alameda   

    AttachedGarageYN  ParkingTotal  BathroomsTotalInteger               City  \
15              True           2.0                    3.0            Menifee   
17              True           2.0                    3.0        Los Angeles   
19              True           2.0                    3.0             Saugus   
20             False           2.0                    5.0  Pacific Palisades   
25              True           2.0                    3.0          Livermore   

    BedroomsTotal  FireplaceYN  LotSizeArea 

In [7]:
print(isinstance(df, pd.DataFrame))  # Should return: True

True


In [13]:
pip install xgboost


   ---------------------------------------- 0.0/149.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/149.9 MB 5.6 MB/s eta 0:00:27
    --------------------------------------- 2.6/149.9 MB 8.9 MB/s eta 0:00:17
   - -------------------------------------- 4.5/149.9 MB 9.0 MB/s eta 0:00:17
   - -------------------------------------- 6.3/149.9 MB 8.8 MB/s eta 0:00:17
   -- ------------------------------------- 8.4/149.9 MB 8.8 MB/s eta 0:00:17
   -- ------------------------------------- 10.0/149.9 MB 8.6 MB/s eta 0:00:17
   --- ------------------------------------ 11.8/149.9 MB 8.6 MB/s eta 0:00:17
   --- ------------------------------------ 13.1/149.9 MB 8.3 MB/s eta 0:00:17
   --- ------------------------------------ 14.9/149.9 MB 8.3 MB/s eta 0:00:17
   ---- ----------------------------------- 16.5/149.9 MB 8.2 MB/s eta 0:00:17
   ---- ----------------------------------- 18.1/149.9 MB 8.1 MB/s eta 0:00:17
   ----- ---------------------------------- 19.7/149.9 MB 8.1 MB

In [15]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Step 1: Load datasets
df_clean = pd.read_csv('cleaned_crmls_dataset.txt', index_col=0)
df_raw = pd.read_csv('crmls_last_6_months.csv')

# Step 2: Filter valid rows in df_clean (non-zero Latitude & Longitude)
df_clean = df_clean[(df_clean['Latitude'] != 0) & (df_clean['Longitude'] != 0)]

# Step 3: Round coordinates for matching
df_clean['lat_round'] = df_clean['Latitude'].round(5)
df_clean['lon_round'] = df_clean['Longitude'].round(5)
df_raw['lat_round'] = df_raw['Latitude'].round(5)
df_raw['lon_round'] = df_raw['Longitude'].round(5)

C:\Users\sarah\AppData\Local\Temp\ipykernel_8484\923518924.py:9: DtypeWarning: Columns (78,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('crmls_last_6_months.csv')


In [17]:
# Step 4: Merge on rounded coordinates to get ClosePrice
df_merged = pd.merge(df_clean, df_raw[['lat_round', 'lon_round', 'ClosePrice']], on=['lat_round', 'lon_round'], how='inner')

# Step 5: Drop helper columns and rows with missing ClosePrice
df_merged.drop(columns=['lat_round', 'lon_round'], inplace=True)
df_merged.dropna(subset=['ClosePrice'], inplace=True)

# Step 6: Prepare data for modeling
X = df_merged.drop(columns=['ClosePrice'])
y = df_merged['ClosePrice']

# Encode categorical columns
for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Fill any remaining NaNs
X = X.fillna(-999)

In [19]:
# Step 7: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 8: Train XGBoost model
model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

# Step 9: Predict and evaluate
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

C:\Users\sarah\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [21]:
# Step 10: Print results
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 435413.18
MAE: 199101.27
R²: 0.7367


In [ ]:
## R² = 0.7367 (Good)
## This means ~74% of the variation in ClosePrice is explained by model.
## In real estate price prediction, an R² above 0.7 is often considered pretty solid, especially given noisy features.

In [ ]:
## MAE = $199K
## On average,  model's prediction is off by $199,101 from the actual price.
## If the average home price in your dataset is around $1 million, this is about a 20% error — reasonable.

In [ ]:
## RMSE = $435K
## RMSE penalizes large errors more than MAE.

## This high value suggests some predictions are very far off (e.g., luxury homes with unpredictable features or mismatched coordinates). 
## So need improvements, such as divide into different groups of price range. NEXT STEP